In [1]:
#!pip -q install "transformers>=4.41.0" "datasets>=2.19.0" \
#               "accelerate>=0.30.0" "peft>=0.11.0" "bitsandbytes>=0.43.1"


In [2]:
from datasets import load_dataset, DatasetDict

train_ds = load_dataset("json", data_files="finance_pc_train.jsonl", split="train")
val_ds   = load_dataset("json", data_files="finance_pc_val.jsonl",   split="train")
data = DatasetDict({"train": train_ds, "validation": val_ds})

print(data)
print(train_ds[0])
print(val_ds[0])


DatasetDict({
    train: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 630
    })
    validation: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 70
    })
})
{'prompt': 'Question: In which note can further details on Legal Proceedings be found within the Consolidated Financial Statements?\nContext: Item 3. Legal Proceedings, which covers litigation and regulatory matters, refers to Note 12 – Commitments and Contingencies for more detailed information within the Consolidated Financial Statements.\nAnswer:', 'completion': ' Further details on Legal Proceedings can be found in Note 12 – Commitments and Contingencies.'}
{'prompt': 'Question: What caused the decrease in Graphics revenue in fiscal year 2023 compared to 2022?\nContext: Graphics - The year-on-year decrease primarily reflects lower sell-in to partners to help reduce channel inventory levels as global macro-economic conditions and COVID-19 related disruptions in China weighed on

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=False)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer loaded.")
print("Pad token:", tokenizer.pad_token)

supports_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
compute_dtype = torch.bfloat16 if supports_bf16 else torch.float16

bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_cfg,
    device_map="auto",
)

print("Model loaded:", MODEL_ID)
print("Compute dtype:", compute_dtype)


Tokenizer loaded.
Pad token: <|eot_id|>


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Model loaded: meta-llama/Meta-Llama-3-8B-Instruct
Compute dtype: torch.bfloat16


In [4]:
MAX_LEN = 1600  ## This we kept 1600 after experimenting

def encode_answer_only(example):
    # training with teacher forcing concept insprred by Ilya Suskever
    full_text = example["prompt"] + example["completion"]

    # tokenize prompt and full sequence
    enc_prompt = tokenizer(
        example["prompt"],
        truncation=True,
        max_length=MAX_LEN,
        padding="max_length",
    )
    enc_full = tokenizer(
        full_text,
        truncation=True,
        max_length=MAX_LEN,
        padding="max_length",
    )

    input_ids = enc_full["input_ids"]
    labels = input_ids.copy()
    pad_id = tokenizer.pad_token_id

    # mask everything except the real answer
    prompt_len = sum(1 for t in enc_prompt["input_ids"] if t != pad_id)
    for i in range(min(prompt_len, len(labels))):
        labels[i] = -100

    # mask padding tokens
    for i, t in enumerate(input_ids):
        if t == pad_id:
            labels[i] = -100

    enc_full["labels"] = labels
    return enc_full

# Map function over both splits
tok_train = train_ds.map(encode_answer_only, batched=False, remove_columns=train_ds.column_names)
tok_val   = val_ds.map(encode_answer_only,   batched=False, remove_columns=val_ds.column_names)

print(tok_train)
print(tok_val)

# display at first row
sample = tok_train[0]
print("Keys:", sample.keys())
print("input_ids[:50]:", sample["input_ids"][:50])
print("labels[:50]:   ", sample["labels"][:50])


Map:   0%|          | 0/630 [00:00<?, ? examples/s]

Map:   0%|          | 0/70 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 630
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 70
})
Keys: dict_keys(['input_ids', 'attention_mask', 'labels'])
input_ids[:50]: [128000, 14924, 25, 763, 902, 5296, 649, 4726, 3649, 389, 25705, 55227, 387, 1766, 2949, 279, 79980, 660, 17961, 70816, 5380, 2014, 25, 5858, 220, 18, 13, 25705, 55227, 11, 902, 14861, 39725, 323, 23331, 13146, 11, 19813, 311, 7181, 220, 717, 1389, 9386, 1392, 323, 2140, 287, 6072, 369]
labels[:50]:    [-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100]


In [5]:
## Test with tokenization output
sample = tok_train[0]
print("FULL TEXT:\n", tokenizer.decode(sample["input_ids"]))
print("\nFULL LABELS:\n", sample["labels"])
answer_ids = [t for t in sample["labels"] if t != -100]
print("\nDECODED ANSWER FROM LABELS:\n", tokenizer.decode(answer_ids))


FULL TEXT:
 <|begin_of_text|>Question: In which note can further details on Legal Proceedings be found within the Consolidated Financial Statements?
Context: Item 3. Legal Proceedings, which covers litigation and regulatory matters, refers to Note 12 – Commitments and Contingencies for more detailed information within the Consolidated Financial Statements.
Answer: Further details on Legal Proceedings can be found in Note 12 – Commitments and Contingencies.<|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|><|eot_id|>

In [6]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
import torch

MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"

supports_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
compute_dtype = torch.bfloat16 if supports_bf16 else torch.float16

bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_cfg,
    device_map="auto",
)

# Prepare for QLoRA
model.gradient_checkpointing_enable()
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

# Attention-LoRA-no MLP adapters. MLP will be added in subsequent experiment
from peft import LoraConfig, get_peft_model, TaskType

lora_cfg = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj","k_proj","v_proj","o_proj"],  # attention only
    bias="none",
)

model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

trainable params: 6,815,744 || all params: 8,037,076,992 || trainable%: 0.0848


In [7]:
#!pip -q install evaluate rouge_score bert_score wandb

In [8]:
import os, wandb
os.environ["WANDB_PROJECT"] = "finance-llama3.0"
wandb.login()

wandb: Currently logged in as: abhi1199 (abhi1199-city-university-of-london) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [9]:
import evaluate
import torch

rouge = evaluate.load("rouge")
bertscore = evaluate.load("bertscore")

# validation
val_prompts = val_ds["prompt"]
val_prompts = [p for p in val_prompts if p is not None]
val_prompts = [p if isinstance(p, str) else str(p) for p in val_prompts]

val_refs = val_ds["completion"]
val_refs = [r for r in val_refs if r is not None]
val_refs = [r if isinstance(r, str) else str(r) for r in val_refs]


pad_id = tokenizer.pad_token_id
prompt_enc = tokenizer(
    val_prompts,
    truncation=True,
    max_length=MAX_LEN,
    padding="max_length",
)

prompt_lens = [sum(1 for t in ids if t != pad_id) for ids in prompt_enc["input_ids"]]


In [10]:
print(type(val_prompts), type(val_prompts[0]))
print(val_prompts[0][:200])


<class 'list'> <class 'str'>
Question: What caused the decrease in Graphics revenue in fiscal year 2023 compared to 2022?
Context: Graphics - The year-on-year decrease primarily reflects lower sell-in to partners to help reduce c


In [11]:
from transformers import TrainerCallback

def compute_metrics_for_generations(eval_pred):
    """
    eval_pred.predictions are generated sequences (input + generated).
    We remove the prompt part using prompt_lens, decode only the generated completion,
    then compute ROUGE and BERTScore against val_refs.
    """
    preds = eval_pred.predictions
    if isinstance(preds, tuple):
        preds = preds[0]

    preds = preds.tolist() if isinstance(preds, torch.Tensor) else preds

    pred_completions = []
    for i, seq in enumerate(preds):
        # strip padding at the right
        if pad_id in seq:
            first_pad = seq.index(pad_id)
            seq = seq[:first_pad]
        # strip the prompt tokens
        start = min(prompt_lens[i], len(seq))
        gen_ids = seq[start:]
        # decode to text
        pred_text = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
        pred_completions.append(pred_text)

    # Align lengths
    n = min(len(pred_completions), len(val_refs))
    preds_n = pred_completions[:n]
    refs_n  = val_refs[:n]

    # ROUGE calcualtion
    rouge_res = rouge.compute(predictions=preds_n, references=refs_n, use_stemmer=True)
    # BERTScore calcualtion
    bert_res = bertscore.compute(predictions=preds_n, references=refs_n, lang="en", model_type="microsoft/deberta-base-mnli")

    return {
        "rouge1": rouge_res.get("rouge1", 0.0),
        "rouge2": rouge_res.get("rouge2", 0.0),
        "rougeL": rouge_res.get("rougeL", 0.0),
        "bertscore_f1": float(sum(bert_res["f1"]) / len(bert_res["f1"])) if bert_res["f1"] else 0.0,
    }

class PerplexityCallback(TrainerCallback):
    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        import math
        if metrics and "eval_loss" in metrics:
            ppl = math.exp(metrics["eval_loss"])

            try:
                wandb.log({"eval_perplexity": ppl}, step=state.global_step)
            except Exception:
                pass


In [15]:
import os, math, torch, wandb, evaluate
from transformers import TrainingArguments, Trainer, TrainerCallback

os.environ["WANDB_PROJECT"] = "finance-llama3.0"
wandb.login()


rouge = evaluate.load("rouge")
bertscore = evaluate.load("bertscore")


val_prompts = [p if isinstance(p, str) else str(p) for p in val_ds["prompt"]]
val_refs    = [c.strip() if isinstance(c, str) else str(c).strip() for c in val_ds["completion"]]

pad_id = tokenizer.pad_token_id
eos_id = tokenizer.eos_token_id
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [16]:
class PerplexityCallback(TrainerCallback):
    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics and "eval_loss" in metrics:
            try:
                wandb.log({"eval_perplexity": math.exp(metrics["eval_loss"])}, step=state.global_step)
            except Exception:
                pass


class GenerateAndScoreCallback(TrainerCallback):
    """
    On each evaluation:
      - generate completions for a subset of validation prompts
      - strip the prompt tokens (using attention mask lengths computed in this batch)
      - compute ROUGE + BERTScore vs gold completions
      - log to W&B
    """
    def __init__(self, tokenizer, val_prompts, val_refs, max_new_tokens=128, n_eval=64):
        self.tok = tokenizer
        self.prompts = val_prompts[:n_eval] if n_eval else val_prompts
        self.refs    = val_refs[:n_eval]    if n_eval else val_refs
        self.max_new = max_new_tokens

    @torch.no_grad()
    def on_evaluate(self, args, state, control, model=None, **kwargs):
        model.eval()

        enc = self.tok(
            self.prompts,
            truncation=True,
            max_length=MAX_LEN,
            padding=True,
            return_tensors="pt",
        )
        enc = {k: v.to(model.device) for k, v in enc.items()}

        # Deterministic decoding for eval
        gen_ids = model.generate(
            **enc,
            max_new_tokens=self.max_new,
            do_sample=False,
            pad_token_id=self.tok.pad_token_id,
            eos_token_id=self.tok.eos_token_id,
        )

        # For each row, compute prompt length from attention mask (batch-true)
        attn = enc["attention_mask"]  # [B, T]
        prompt_lens = attn.sum(dim=1).tolist()


        preds = []
        for i, seq in enumerate(gen_ids.tolist()):

            if pad_id in seq:
                seq = seq[:seq.index(pad_id)]
            start = min(int(prompt_lens[i]), len(seq))
            comp_ids = seq[start:]
            preds.append(self.tok.decode(comp_ids, skip_special_tokens=True).strip())

        # Compute metrics (lowercase for ROUGE stability)
        preds_lc = [p.lower() for p in preds]
        refs_lc  = [r.lower() for r in self.refs]

        rouge_res = rouge.compute(predictions=preds_lc, references=refs_lc, use_stemmer=True)
        bert_res  = bertscore.compute(predictions=preds, references=self.refs, lang="en",
                                      model_type="microsoft/deberta-base-mnli",
                                      rescale_with_baseline=True)

        metrics = {
            "rouge1": rouge_res.get("rouge1", 0.0),
            "rouge2": rouge_res.get("rouge2", 0.0),
            "rougeL": rouge_res.get("rougeL", 0.0),
            "bertscore_f1": float(sum(bert_res["f1"]) / len(bert_res["f1"])) if bert_res["f1"] else 0.0,
        }

        try:
            wandb.log(metrics, step=state.global_step)
        except Exception:
            pass


In [20]:
## training arguments
supports_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8

args = TrainingArguments(
    output_dir="llama3-8b-fin-qlora-attnOnly",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=50,

    eval_strategy="steps",
    eval_steps=200,
    save_steps=200,
    save_total_limit=2,

    bf16=supports_bf16,
    fp16=not supports_bf16,
    gradient_checkpointing=True,

    report_to=["wandb"],
    run_name="llama3-8b-fin-qlora-r8-attnOnly",
)


In [21]:
# callback
callbacks = [
    PerplexityCallback(),
    GenerateAndScoreCallback(
        tokenizer=tokenizer,
        val_prompts=val_prompts,
        val_refs=val_refs,
        max_new_tokens=128,
        n_eval=64,
    ),
]

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tok_train,
    eval_dataset=tok_val,
    callbacks=callbacks,
)

trainer.train()
final = trainer.evaluate()
print(final)
print("Final Perplexity:", math.exp(final["eval_loss"]))


Step,Training Loss,Validation Loss


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/557M [00:00<?, ?B/s]

{'eval_loss': 0.4334127902984619, 'eval_runtime': 67.0153, 'eval_samples_per_second': 1.045, 'eval_steps_per_second': 1.045, 'epoch': 1.0}
Final Perplexity: 1.5425128236286836


In [26]:
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"


In [27]:
@torch.no_grad()
def generate_completions(model, tokenizer, prompts, max_new=128, batch_size=8):
    pad_id = tokenizer.pad_token_id
    model.eval()
    outs = []

    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i+batch_size]
        enc = tokenizer(batch, truncation=True, max_length=MAX_LEN - max_new,
                        padding=True, return_tensors="pt")
        enc = {k: v.to(model.device) for k, v in enc.items()}

        gen = model.generate(
            **enc,
            max_new_tokens=max_new,
            do_sample=False,
            pad_token_id=pad_id,
            eos_token_id=tokenizer.eos_token_id,
        )

        attn = enc["attention_mask"]
        lens = attn.sum(dim=1).tolist()

        for j, seq in enumerate(gen.tolist()):
            if pad_id in seq:
                seq = seq[:seq.index(pad_id)]
            start = min(int(lens[j]), len(seq))
            comp_ids = seq[start:]
            outs.append(tokenizer.decode(comp_ids, skip_special_tokens=True).strip())

    return outs


In [29]:
full_preds = generate_completions(model, tokenizer, val_prompts, max_new=128, batch_size=8)

# Lowercase for fair matching
rouge_full = rouge.compute(
    predictions=[p.lower() for p in full_preds],
    references=[r.lower() for r in val_refs],
    use_stemmer=True
)

bert_full = bertscore.compute(
    predictions=full_preds,
    references=val_refs,
    lang="en",
    model_type="microsoft/deberta-base-mnli",
    rescale_with_baseline=True
)

import wandb
wandb.log({
    "final_rouge1": rouge_full.get("rouge1", 0.0),
    "final_rouge2": rouge_full.get("rouge2", 0.0),
    "final_rougeL": rouge_full.get("rougeL", 0.0),
    "final_bertscore_f1": float(sum(bert_full["f1"]) / len(bert_full["f1"])),
})



The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.

In [31]:
@torch.no_grad()
def ask_question(model, tokenizer, question, context=None, max_new_tokens=128):
    model.eval()

    # Format the prompt
    if context:
        prompt = f"Question: {question}\nContext: {context}\nAnswer:"
    else:
        prompt = f"Question: {question}\nAnswer:"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Generate response
    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,  # Deterministic output
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    # Strip prompt part and decode only generated answer
    generated = output[0][inputs['input_ids'].shape[1]:]
    answer = tokenizer.decode(generated, skip_special_tokens=True).strip()
    return answer


In [34]:
question = "In which note can further details on Legal Proceedings be found within the Consolidated Financial Statements?"
context = ""

response = ask_question(model, tokenizer, question, context)
print("Answer:", response)


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Answer: Note 14: Legal Proceedings. Further details on Legal Proceedings can be found in Note 14 of the Consolidated Financial Statements. This note provides information on the Company's involvement in various legal proceedings and the potential impact on the Company's financial condition and results of operations. It also includes information on the Company's exposure to certain legal proceedings and the Company's involvement in certain legal proceedings. Additionally, this note provides information on the Company's legal and regulatory compliance, including information on the Company's compliance with applicable laws and regulations. It also includes information on the Company's legal and regulatory compliance, including information on the Company's compliance with applicable laws


In [35]:
question = "What was the increase in the consolidated operating profit for 2023 compared to 2022?"
context = ""

response = ask_question(model, tokenizer, question, context)
print("Answer:", response)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Answer: The consolidated operating profit for 2023 increased by 12.4% compared to 2022. This was primarily due to higher sales and improved operating margins. However, the increase was partially offset by higher operating expenses. The consolidated operating profit for 2023 was $1.4 billion, compared to $1.2 billion in 2022. The consolidated operating profit for 2023 was $1.4 billion, compared to $1.2 billion in 2022. The consolidated operating profit for 2023 was $1.4 billion, compared to $1.2 billion in 2022. The consolidated
